# Part 3 — BERT-Style Robust NLP (RunPod GPU)

**Upload `part3_BERT_Robust_NLP_Experiments.py` next to this notebook.**

## Checklist before running
| Step | Action |
|------|--------|
| 1 | Upload **this notebook** + `part3_BERT_Robust_NLP_Experiments.py` to the same folder (e.g. `/workspace/robustNN/`) |
| 2 | Set `HF_TOKEN` in RunPod UI → Pod → Environment **OR** uncomment the line in Cell 3 |
| 3 | Run all cells top-to-bottom (Kernel → Restart & Run All) |
| 4 | **Before stopping the pod**: run Cell 8 to zip & preserve all results |

## Outputs
All CSVs and PNGs are saved to `ROBUST_NN_WORKSPACE/results_bert/`  
(default: `/workspace/runpod_outputs/results_bert/`)

> **Security**: Never commit your HF token into a notebook or commit to GitHub.


In [ ]:
# ── Cell 1: Install pinned packages (RunPod) ─────────────────────────────────
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "--root-user-action=ignore", *pkgs]
    )

pip_install([
    "typing_extensions>=4.10.0",
    "torch>=2.6.0", "torchvision>=0.21.0",
    "transformers>=4.46.0",
    "datasets>=2.20.0",
    "accelerate>=0.33.0",
    "huggingface_hub>=0.24.0",
    "scikit-learn>=1.3.0",
    "matplotlib>=3.8.0",
    "seaborn>=0.13.0",
    "pandas>=2.1.0",
    "tqdm>=4.66.0",
    "plotly>=5.22.0",
    "kaleido>=0.2.1",
])
print("✓ pip install done.", sys.version)
print("If this is a fresh install, restart kernel once before continuing.")


In [ ]:
# ── Cell 2: Runtime sanity + GPU check ───────────────────────────────────────
import importlib.metadata as md

def ver(pkg):
    try:
        return md.version(pkg)
    except Exception:
        return "not-installed"

print("typing_extensions:", ver("typing_extensions"))
print("torch:", ver("torch"))
print("transformers:", ver("transformers"))
print("datasets:", ver("datasets"))

import torch
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f"✓ GPU : {dev.name}  VRAM = {dev.total_memory / 1024**3:.1f} GB")
    print(f"  CUDA: {torch.version.cuda}  PyTorch: {torch.__version__}")
else:
    print("⚠ No GPU found — CPU-only mode will be very slow for BERT fine-tuning.")


In [ ]:
# ── Cell 3: Workspace & HF token setup ────────────────────────────────────────
# HF_TOKEN: reads from the RunPod environment variable — NEVER paste tokens here.
# If you did NOT set it in RunPod UI, uncomment and edit the line below:
# import os; os.environ["HF_TOKEN"] = "hf_..."   # ← temporary, do not commit!

import os
from pathlib import Path

WORKSPACE = os.environ.get("ROBUST_NN_WORKSPACE", "/workspace/runpod_outputs")
Path(WORKSPACE).mkdir(parents=True, exist_ok=True)
os.environ["ROBUST_NN_WORKSPACE"]      = WORKSPACE
os.environ.setdefault("ROBUST_NN_RESULTS_SUBDIR", "results_bert")
os.environ.setdefault("HF_HOME",            str(Path(WORKSPACE) / "hf_home"))
os.environ.setdefault("TRANSFORMERS_CACHE", str(Path(WORKSPACE) / "transformers_cache"))

print("WORKSPACE         :", WORKSPACE)
print("Results subfolder :", os.environ["ROBUST_NN_RESULTS_SUBDIR"])
print("HF_TOKEN set      :", "HF_TOKEN" in os.environ and bool(os.environ.get("HF_TOKEN")))


In [ ]:
# ── Cell 4: Locate companion .py script ───────────────────────────────────────
from pathlib import Path

def find_script(name: str) -> Path:
    cwd = Path.cwd().resolve()
    search_paths = [
        cwd,
        cwd.parent,
        Path("/workspace"),
        Path("/workspace/robustNN"),
        Path("/workspace/Robust-NN-learning"),
    ]
    for base in search_paths:
        p = base / name
        if p.is_file():
            return p
    raise FileNotFoundError(
        f"Cannot find '{name}'. Upload it to the same folder as this notebook.\n"
        f"Searched: {[str(b) for b in search_paths]}"
    )

SCRIPT = find_script("part3_BERT_Robust_NLP_Experiments.py")
print("✓ Script found:", SCRIPT)


In [ ]:
# ── Cell 5: RUN the full experiment pipeline ───────────────────────────────────
# Batteries A–D: clean baseline → uniform noise → class-dep noise → GCE(q) sweep.
# QUICK_RUN=True (default in the .py) takes ~30-60 min on L4/L40.
# Each battery saves its own CSV immediately; results survive partial runs.
import runpy
runpy.run_path(str(SCRIPT), run_name="__main__")


## After the run: inspect results & download
Run the cells below to see what was saved, preview the summary table,
and create a zip archive for download.  
**Run the zip cell BEFORE stopping the pod** — RunPod storage is wiped on shutdown.


In [ ]:
# ── Cell 6: List saved output files ────────────────────────────────────────────
import os
from pathlib import Path

ws  = Path(os.environ["ROBUST_NN_WORKSPACE"])
sub = os.environ.get("ROBUST_NN_RESULTS_SUBDIR", "results_bert")
rd  = ws / sub

if rd.exists():
    files = sorted(rd.iterdir())
    print(f"Results in {rd}  ({len(files)} files):\n")
    for f in files:
        print(f"  {f.name:45s}  {f.stat().st_size/1024:7.1f} KB")
else:
    print("Results folder not yet created — has Cell 5 completed?")


In [ ]:
# ── Cell 7: Preview summary accuracy table ─────────────────────────────────────
import pandas as pd, os
from pathlib import Path

summary_path = (
    Path(os.environ["ROBUST_NN_WORKSPACE"])
    / os.environ.get("ROBUST_NN_RESULTS_SUBDIR", "results_bert")
    / "summary_all.csv"
)
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print(f"Summary table  ({len(df)} rows × {len(df.columns)} cols)\n")
    pd.set_option("display.max_rows", 60)
    pd.set_option("display.width", 120)
    print(df[["dataset", "loss", "noise_type", "noise_rate", "acc_str"]].to_string(index=False))
else:
    print("summary_all.csv not yet generated — run Cell 5 first.")


In [ ]:
# ── Cell 8 (FINAL): Zip entire workspace for download ─────────────────────────
# Run this BEFORE shutting down the pod — RunPod deletes disks on pod termination.
import shutil, os
from pathlib import Path

ws  = Path(os.environ["ROBUST_NN_WORKSPACE"])
out = ws.parent / (ws.name + "_part3_archive")
arc = shutil.make_archive(str(out), "zip", root_dir=ws.parent, base_dir=ws.name)
size_mb = Path(arc).stat().st_size / 1024**2
print(f"✓ Archive created : {arc}")
print(f"  Size            : {size_mb:.1f} MB")
print("  → Download this file from the RunPod file browser before stopping the pod.")
